In [2]:
import os, sys, argparse, json
from glob import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def normalize_col(name:str)->str:
    key = name.strip().lower().replace(" ", "_").replace("-", "_")
    key = key.replace("(", "").replace(")", "")
    return key

def find_required_columns(cols):
    norm = {normalize_col(c): c for c in cols}
    num_candidates = ["num_genomes", "n_genomes", "numgenomes", "genomes", "n"]
    bit_candidates = [
        "bitting_rate", "biting_rate", "bitrate", "bit_rate", "bite_rate",
        "bittingrate", "bitingrate"
    ]
    found = {}
    for k in num_candidates:
        if k in norm:
            found["num_genomes"] = norm[k]
            break
    for k in bit_candidates:
        if k in norm:
            found["bitting_rate"] = norm[k]
            break
    return found

def load_any(path):
    ext = os.path.splitext(path)[1].lower()
    if ext == ".csv":
        return pd.read_csv(path)
    if ext in (".tsv", ".tab"):
        return pd.read_csv(path, sep="\t")
    if ext in (".parquet", ".pq"):
        return pd.read_parquet(path)
    if ext == ".json":
        try:
            return pd.read_json(path, lines=True)
        except:
            return pd.read_json(path)
    if ext in (".xlsx", ".xls"):
        return pd.read_excel(path)
    return None

def numeric_metric_columns(df, exclude):
    cols = []
    for c in df.columns:
        if c in exclude:
            continue
        if pd.api.types.is_numeric_dtype(df[c]):
            cols.append(c)
    return cols

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--input", "-i", required=True, help="Carpeta con archivos de resultados")
    ap.add_argument("--output", "-o", required=True, help="Carpeta de salida")
    args = ap.parse_args()

    in_dir = args.input
    out_dir = args.output
    os.makedirs(out_dir, exist_ok=True)

    patterns = ["**/*.csv","**/*.tsv","**/*.tab","**/*.parquet","**/*.pq","**/*.json","**/*.xlsx","**/*.xls"]
    files = []
    for p in patterns:
        files.extend(glob(os.path.join(in_dir, p), recursive=True))
    files = sorted(set(files))

    combined_rows = []
    log_rows = []

    for f in files:
        try:
            df = load_any(f)
            if df is None or len(df)==0:
                log_rows.append({"file": f, "status": "skipped (empty or unsupported)"})
                continue
            mapping = find_required_columns(df.columns)
            if set(mapping.keys()) != {"num_genomes","bitting_rate"}:
                log_rows.append({"file": f, "status": "missing required columns", "columns": list(df.columns)})
                continue
            metrics = numeric_metric_columns(df, exclude=[mapping["num_genomes"], mapping["bitting_rate"]])
            if not metrics:
                log_rows.append({"file": f, "status": "no numeric metrics to compare", "columns": list(df.columns)})
                continue
            melt = df[[mapping["num_genomes"], mapping["bitting_rate"]] + metrics].copy()
            melt = melt.melt(
                id_vars=[mapping["num_genomes"], mapping["bitting_rate"]],
                value_vars=metrics,
                var_name="metric",
                value_name="value",
            )
            melt.rename(columns={
                mapping["num_genomes"]: "num_genomes",
                mapping["bitting_rate"]: "bitting_rate"
            }, inplace=True)
            melt["source_file"] = os.path.relpath(f, in_dir)
            combined_rows.append(melt)
            log_rows.append({"file": f, "status": "loaded", "metrics_found": metrics})
        except Exception as e:
            log_rows.append({"file": f, "status": f"error: {e}"})

    if combined_rows:
        combined = pd.concat(combined_rows, ignore_index=True)
        combined.to_csv(os.path.join(out_dir, "combined_results_long.csv"), index=False)

        # Una pivote por cada métrica
        for metric_name, g in combined.groupby("metric"):
            pv = g.pivot_table(index="num_genomes", columns="bitting_rate", values="value", aggfunc="mean")
            # ordenar si son numéricos
            try:
                pv.index = pd.to_numeric(pv.index, errors="ignore")
                pv.sort_index(inplace=True)
            except:
                pass
            try:
                pv.columns = pd.to_numeric(pv.columns, errors="ignore")
                pv = pv.reindex(sorted(pv.columns, key=lambda x: (isinstance(x, str), x)), axis=1)
            except:
                pass

            pv_path = os.path.join(out_dir, f"pivot_{metric_name}.csv")
            pv.to_csv(pv_path)

            # Heatmap con matplotlib (sin especificar colores)
            fig = plt.figure()
            arr = pv.values.astype(float)
            im = plt.imshow(arr, aspect='auto')
            plt.colorbar(im)
            plt.xticks(range(pv.shape[1]), pv.columns, rotation=45, ha='right')
            plt.yticks(range(pv.shape[0]), pv.index)
            plt.title(f"Heatmap: {metric_name} (rows=num_genomes, cols=bitting_rate)")
            plt.xlabel("bitting_rate")
            plt.ylabel("num_genomes")
            plt.tight_layout()
            fig_path = os.path.join(out_dir, f"heatmap_{metric_name}.png")
            plt.savefig(fig_path, dpi=150)
            plt.close(fig)

    # Guardar log
    pd.DataFrame(log_rows).to_csv(os.path.join(out_dir, "ingestion_log.csv"), index=False)
    print(f"Procesado. Archivos analizados: {len(files)}. Revisa la carpeta de salida: {out_dir}")
